In [ ]:
import httpx
from bs4 import BeautifulSoup
import hashlib
import json
import os
import time
from datetime import datetime

class VibloScraper:
    def __init__(self, target_count=2000):
        self.base_url = "https://viblo.asia"
        self.api_url = "https://viblo.asia/newest"
        self.target_count = target_count
        self.headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"
        }
        self.output_path = "data/raw/viblo_data.jsonl"
        self.scraped_urls = set() # Tránh cào trùng bài
        
        os.makedirs(os.path.dirname(self.output_path), exist_ok=True)

    def generate_doc_id(self, url):
        return hashlib.md5(url.encode('utf-8')).hexdigest()

    def get_article_urls(self, client, page):
        """Lấy danh sách link bài viết từ trang danh sách"""
        urls = []
        try:
            response = client.get(f"{self.api_url}?page={page}", timeout=15)
            if response.status_code != 200:
                return urls
            
            soup = BeautifulSoup(response.text, 'html.parser')
            # Dung selector theo pattern URL de giam phu thuoc vao ten class giao dien
            links = soup.select("a[href^='/p/']")
            seen = set()
            for link in links:
                href = link.get('href')
                if href and href.startswith('/p/'):
                    full_url = self.base_url + href
                    if full_url not in self.scraped_urls and full_url not in seen:
                        urls.append(full_url)
                        seen.add(full_url)
        except Exception as e:
            print(f"⚠️ Lỗi khi quét trang danh sách {page}: {e}")
        return urls

    def scrape_content(self, client, url):
        """Cào và làm sạch nội dung chi tiết bài viết"""
        try:
            response = client.get(url, timeout=15)
            if response.status_code != 200:
                return None

            soup = BeautifulSoup(response.text, 'html.parser')
            
            title_node = soup.find('h1', class_='article-content__title')
            # Viblo nội dung nằm trong class này
            content_node = soup.find('div', class_='article-content__body')
            
            if not title_node or not content_node:
                return None

            # Loại bỏ code blocks nếu bạn chỉ muốn lấy văn bản (tùy chọn)
            # Nếu muốn giữ code thì comment dòng dưới lại
            for code in content_node.find_all(['pre', 'code']):
                code.decompose()

            # Loại bỏ các thành phần rác khác
            for tag in content_node(["script", "style", "aside", "iframe", "button"]):
                tag.decompose()

            return {
                "doc_id": self.generate_doc_id(url),
                "url": url,
                "title": title_node.get_text(strip=True),
                "content": content_node.get_text(separator="\n", strip=True),
                "source": "viblo",
                "timestamp": datetime.now().isoformat()
            }
        except Exception as e:
            print(f"❌ Lỗi khi xử lý bài {url}: {e}")
            return None

    def run(self):
        count = 0
        page = 1
        print(f"🚀 Bắt đầu chiến dịch cào {self.target_count} bài báo...")

        # Sử dụng Client session để tối ưu hiệu suất
        with httpx.Client(headers=self.headers) as client:
            with open(self.output_path, 'a', encoding='utf-8') as f:
                while count < self.target_count:
                    print(f"\n--- Đang quét trang: {page} ---")
                    article_links = self.get_article_urls(client, page)
                    
                    if not article_links:
                        print("Không còn bài viết mới. Dừng cào.")
                        break

                    for link in article_links:
                        if count >= self.target_count:
                            break
                        
                        # Nghỉ ngắn để tránh bị block
                        time.sleep(0.5) 
                        
                        data = self.scrape_content(client, link)
                        if data:
                            f.write(json.dumps(data, ensure_ascii=False) + "\n")
                            self.scraped_urls.add(link)
                            count += 1
                            if count % 10 == 0:
                                print(f"✅ Đã thu thập: {count}/{self.target_count} bài.")

                    page += 1

        print(f"\n rực rỡ! Đã lưu {count} bài vào: {self.output_path}")

if __name__ == "__main__":
    # Khởi tạo và chạy
    scraper = VibloScraper(target_count=2000)
    scraper.run()

🚀 Bắt đầu chiến dịch cào 2000 bài báo...

--- Đang quét trang: 1 ---
Không còn bài viết mới. Dừng cào.

 rực rỡ! Đã lưu 0 bài vào: data/raw/viblo_data.jsonl


In [2]:
%pip install httpx
 

  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
Using cached httpx-0.28.1-py3-none-any.whl (73 kB)
Using cached httpcore-1.0.9-py3-none-any.whl (78 kB)
   ---------------------------------------- 0.0/113.6 kB ? eta -:--:--
   --- ------------------------------------ 10.2/113.6 kB ? eta -:--:--
   --- ------------------------------------ 10.2/113.6 kB ? eta -:--:--
   ---------- ---------------------------- 30.7/113.6 kB 325.1 kB/s eta 0:00:01
   --------------------- ----------------- 61.4/113.6 kB 469.7 kB/s eta 0:00:01
   -------------------------------------- 113.6/113.6 kB 662.5 kB/s eta 0:00:00
Using cached h11-0.16.0-py3-none-any.whl (37 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
